# Design Spec — `spur-flight-gateway`
## A SQL-over-anything adapter framework (REST / GraphQL / web → Arrow Flight → DuckDB airport)

**Date:** 2026-05-31  •  **Status:** ✅ Approved (decisions locked §17)  •  **Owner:** notebook team

> Turn any REST API, GraphQL endpoint, or web service into a SQL-queryable datasource by implementing a small **adapter**. Users query with plain SQL in a SPUR notebook cell; the gateway translates those queries into upstream API calls and streams results back as Apache Arrow. **Polymarket** is adapter #1.

This spec is **grounded** against the current worktree via the `code_*` graph tools. Every integration touchpoint cites a real `file:line` and stable symbol id (see the *Grounding Appendix*).

## 1. Executive summary

- Build a **read-only Arrow Flight server**, living as a **sub-crate under `spur-notebook`** (`crates/spur-notebook/flight-gateway`, package `spur-flight-gateway`), that implements the DuckDB **airport** extension's server contract.
- DuckDB + the airport community extension **already exist** and do the heavy lifting (SQL engine, Flight client, predicate generation, JSON handling on the client). **We only build the server side**: an adapter framework that answers airport's RPCs by calling upstream APIs.
- **Adapter model is C3 (hybrid):** a declarative **manifest** for clean REST tables (80% case) + a Rust **`Adapter` trait** escape hatch for signing / GraphQL / cursor pagination / parameterized endpoints (exposed as airport **table-valued functions**).
- **Scope cut for v1:** implement only airport's **read subset** (catalog discovery, table scan, predicate + projection pushdown, TVFs). Skip DML / DDL / transactions / time-travel entirely.
- **Integration:** a new `Airport` variant on `DatasourceKind`, plus a non-path **locator** (`source` + `gateway_addr`), wired through the existing `attach_datasource` / `reconcile_open_datasource_catalog` / `introspect_datasource` seam in `spur-notebook`.
- **Proof:** `SELECT question, volume FROM polymarket.markets WHERE active = true` (manifest + pushdown) and `SELECT * FROM polymarket.orderbook('0x…', depth => 50)` (TVF + trait).

## 2. Goals / non-goals

### Goals
1. Query arbitrary REST/GraphQL APIs as SQL tables from a notebook cell, with no per-query glue code.
2. Make adding a new source cheap: declarative manifest for the common case; a small trait impl for the hard case.
3. Push WHERE-clause predicates and column projection **down to the API** as query params when the API supports it.
4. Keep upstream API credentials **server-side only** — never returned to the SQL client.
5. Co-locate in the Rust monorepo, lifecycle-managed by the notebook backend (no separate deployable in v1).

### Non-goals (v1)
- No writes: INSERT / UPDATE / DELETE / CREATE / ALTER / transactions / time-travel (airport supports them; we don't need them).
- No zero-recompile (hot-loaded) adapters — trait adapters compile into the crate. (Deferred; see *Phasing*.)
- No multi-tenant credential isolation, no column-statistics pushdown, no standalone distributable binary. (Deferred.)
- Not a generic Flight **SQL** server — airport is **plain Flight**, so Flight SQL clients are explicitly out of scope.

## 3. Background — the airport reality (the finding that shaped this design)

The DuckDB **airport** extension is **not** a Flight SQL client. It speaks **raw Arrow Flight** with airport-specific conventions layered on top. This is why off-the-shelf Flight **SQL** gateways (Spice.ai OSS, ROAPI, `datafusion-flight-sql-server`) **cannot** be `ATTACH`-ed by airport without an adapter shim — so we implement the airport wire contract directly.

| Airport convention | Detail |
|---|---|
| Transport | Arrow Flight over gRPC (`grpc://` / `grpc+tls://`) |
| Catalog | `DoAction` with named actions (`list_schemas`, table discovery, `catalog_version`, …) — bodies **MessagePack**-encoded |
| Tickets | `GetFlightInfo` returns a **MessagePack** ticket `{ table, filter, projection, tvf_args }`; consumed by `DoGet` |
| Predicate pushdown | DuckDB sends a **JSON-encoded predicate expression** in `ScanOptions.Filter`; the server translates it to API params |
| Schema | Arrow IPC bytes in `GetFlightInfo` / `GetSchema` |
| Auth | `authorization: Bearer <token>` in gRPC metadata |
| Reference impls | `airport-go` (Go, authoritative spec we port from) and `python-flight-server` (Python) |

**Feature surface airport exposes** (from <https://airport.query.farm/features.html>): custom scalar functions on the server, **table-valued functions with arguments**, predicate pushdown, column statistics, RowId pseudocolumn, INSERT/UPDATE/DELETE, CREATE/DROP/ALTER, transactions, time-travel, secret-manager integration. **v1 uses only the read-relevant subset of these.**

Sources: <https://airport.query.farm/> · <https://airport.query.farm/features.html> · `airport-go` <https://pkg.go.dev/github.com/hugr-lab/airport-go> · `python-flight-server` <https://github.com/Query-farm/python-flight-server>

## 4. System context

Where the gateway sits relative to the existing notebook, DuckDB, and the outside world. Everything inside the **SPUR notebook process** is ours; DuckDB + airport are reused; the gateway is the only new long-lived component.

```mermaid
flowchart LR
  subgraph User["Notebook UI (jute / Tauri)"]
    cell["SQL cell:\nSELECT ... FROM polymarket.markets"]
    catalog["Datasource catalog panel"]
  end

  subgraph Proc["SPUR notebook process (Rust)"]
    direction TB
    kernel["Python kernel\n(query execution)"]
    duck[("DuckDB\n+ airport extension")]
    daemon["NotebookDaemonControl\n(spur-notebook/src/mcp/mod.rs)"]
    subgraph GW["spur-flight-gateway (NEW sub-crate)"]
      flight["Arrow Flight server\n(tonic, loopback)"]
      reg["AdapterRegistry"]
      adp["Adapters\n(manifest + trait)"]
    end
  end

  subgraph World["World data APIs"]
    poly[["Polymarket\nGamma / CLOB REST"]]
    other[["any REST / GraphQL\nendpoint"]]
  end

  cell --> kernel --> duck
  duck -- "ATTACH (TYPE airport)\ngrpc://127.0.0.1:PORT" --> flight
  catalog <-- "list_datasources" --> daemon
  daemon -- "spawn + lifecycle" --> flight
  flight --> reg --> adp
  adp -- "reqwest: GET/POST\n+ params, pagination, auth" --> poly
  adp --> other
  poly -- "JSON" --> adp
  adp -- "Arrow RecordBatch stream (DoGet)" --> duck
```

## 5. Crate & module structure

The workspace already nests a crate under `spur-notebook` (`jute-notebook/src-tauri`, a workspace member). The gateway follows that precedent as `crates/spur-notebook/flight-gateway`.

```mermaid
flowchart TB
  subgraph crate["crates/spur-notebook/flight-gateway  (package: spur-flight-gateway)"]
    lib["lib.rs\nserver bootstrap + public API\n(GatewayHandle::spawn / shutdown)"]
    subgraph airport["airport/  — the wire contract (read subset)"]
      svc["service.rs\nimpl FlightService:\nGetFlightInfo . DoGet . ListFlights\nDoAction . GetSchema . Handshake"]
      cat["catalog.rs\nlist_schemas / table enum\nDoAction (MessagePack)"]
      tik["ticket.rs\nMessagePack ticket\n{table,filter,projection,tvf_args}\n(ported from airport-go)"]
      flt["filter.rs\nairport JSON predicate grammar\n-> typed Predicate\n(ported from airport-go)"]
    end
    subgraph adapter["adapter/  — the framework"]
      atr["mod.rs\ntrait Adapter + AdapterRegistry"]
      man["manifest.rs\nserde manifest model"]
      mad["manifest_adapter.rs\nimpl Adapter for ManifestAdapter"]
      http["http.rs\nreqwest client:\npagination . rate-limit . retry . cache"]
      aj["arrow_json.rs\nJSON/GraphQL -> RecordBatch\n(declared schema)"]
    end
    subgraph adapters["adapters/  — concrete sources"]
      pm["polymarket.rs\nadapter #1"]
    end
    sec["secrets.rs\nper-source credential resolution"]
  end

  lib --> svc
  svc --> cat & tik & flt
  svc --> atr
  atr --> man & http
  man --> mad --> http --> aj
  atr --> sec
  pm --> man
  pm -. "trait impl for orderbook TVF" .-> atr
```

## 6. The airport contract we implement (read subset only)

| RPC / DoAction | v1 implements | v1 skips |
|---|---|---|
| `Handshake` | yes — Bearer-token mint + validate | — |
| `DoAction: list_schemas`, table discovery, `catalog_version` | yes — catalog built from registered adapters | `create_*`, `drop_*`, `alter_*`, `add_column`, `create_transaction` |
| `GetFlightInfo` / `GetSchema` | yes — declared Arrow schema + MessagePack ticket | — |
| `DoGet` | yes — stream `RecordBatch`es from adapter | — |
| `ListFlights` | yes — powers `airport_list_flights()` | — |
| Predicate + projection pushdown | yes — parsed, handed to adapter | column statistics |
| Table-valued functions (args) | yes — args ride in the ticket | DML / DDL / time-travel / RowId |

The ticket binary format and the JSON predicate grammar are the only **underdocumented** pieces — they are **ported from `airport-go`** and isolated in `ticket.rs` / `filter.rs` behind golden round-trip tests.

## 7. Query lifecycle (sequence)

End-to-end for `SELECT question, volume FROM polymarket.markets WHERE active = true ORDER BY volume DESC LIMIT 10`.

```mermaid
sequenceDiagram
  participant Cell as Notebook SQL cell
  participant Duck as DuckDB + airport
  participant Flight as Gateway FlightService
  participant Reg as AdapterRegistry
  participant Adp as ManifestAdapter (polymarket)
  participant API as Polymarket REST

  Cell->>Duck: SELECT ... WHERE active=true LIMIT 10
  Note over Duck: airport already ATTACHed (grpc://127.0.0.1:PORT)
  Duck->>Flight: GetFlightInfo(path=[polymarket,markets], filter JSON: active=true, projection=[question,volume])
  Flight->>Flight: filter.rs parse JSON -> Predicate
  Flight-->>Duck: FlightInfo{ schema(IPC), ticket(msgpack) }
  Duck->>Flight: DoGet(ticket)
  Flight->>Reg: route table markets
  Reg->>Adp: scan(ScanRequest{predicates, projection})
  Adp->>Adp: map predicate -> ?active=true ; pagination plan
  loop each page (until cursor exhausted)
    Adp->>API: GET /markets?active=true&limit=500&offset=N
    API-->>Adp: JSON page
    Adp->>Adp: arrow_json: JSON -> RecordBatch (declared schema)
    Adp-->>Flight: RecordBatch
    Flight-->>Duck: FlightData (stream)
  end
  Duck->>Duck: apply residual filter / ORDER BY / LIMIT
  Duck-->>Cell: result rows
```

## 8. Predicate pushdown & TVF argument flow

Two paths converge on `Adapter::scan`: **table + WHERE** (manifest pushdown) and **TVF with args** (trait). Predicates the API can't express are returned to DuckDB to apply as a residual filter (correctness is never sacrificed for pushdown).

```mermaid
flowchart TB
  start(["airport ticket (MessagePack)"]) --> dec["ticket.rs: decode"]
  dec --> br{"table scan or TVF?"}

  br -- "table scan" --> pred["filter.rs: JSON predicate -> Vec of Predicate"]
  pred --> map{"manifest table.filters maps column -> param?"}
  map -- yes --> q1["push down: append ?param=value"]
  map -- no --> res["residual: keep predicate, DuckDB re-applies after DoGet"]

  br -- "TVF" --> args["tvf_args -> path/query params (orderbook token_id, depth)"]

  q1 --> http["http.rs: build request + pagination + auth + cache"]
  res --> http
  args --> http
  http --> page["paginate upstream"]
  page --> conv["arrow_json: page -> RecordBatch (declared schema, typed coercion)"]
  conv --> stream(["DoGet RecordBatch stream"])
```

## 9. Adapter model (C3) — manifest + trait

### 9a. Declarative manifest (the 80% — clean REST tables)

```toml
[source]                 # -> one airport schema/catalog
name = "polymarket"
base_url = "https://gamma-api.polymarket.com"
auth = "none"
pagination = { style = "offset", limit_param = "limit", offset_param = "offset", page_size = 500 }

[[table]]
name = "markets"
path = "/markets"
[table.columns]          # JSON path -> (column, Arrow type)
id       = { json = "$.id",       type = "Utf8" }
question = { json = "$.question", type = "Utf8" }
active   = { json = "$.active",   type = "Boolean" }
volume   = { json = "$.volume",   type = "Float64" }
[table.filters]          # SQL predicate column -> query param (enables WHERE pushdown)
active = { param = "active" }
```

### 9b. Code escape hatch (the hard 20% — signing, GraphQL, cursor pagination, TVFs)

```rust
#[async_trait]
pub trait Adapter: Send + Sync {
    /// Tables + table-valued functions, each with a declared Arrow schema.
    fn catalog(&self) -> Vec<TableDef>;
    /// Stream Arrow batches for one scan / TVF call.
    async fn scan(&self, req: ScanRequest) -> Result<BoxStream<'static, Result<RecordBatch>>>;
}

pub struct ScanRequest {
    pub table: String,
    pub predicates: Vec<Predicate>,   // from filter.rs (already typed)
    pub projection: Option<Vec<String>>,
    pub tvf_args: Vec<ScalarValue>,   // for table-valued functions
    pub auth: ResolvedAuth,           // from secrets.rs, server-side only
}
```

`ManifestAdapter` is *just one* `impl Adapter`. Polymarket uses the manifest for `markets` / `events` and a ~20-line trait impl for the `orderbook(token_id, depth)` TVF. **The framework owns** HTTP, pagination, rate-limiting, caching, JSON->Arrow, and the entire wire protocol; an adapter author only describes endpoints and (optionally) shapes requests/responses.

```mermaid
classDiagram
  class Adapter {
    <<trait>>
    +catalog() Vec~TableDef~
    +scan(ScanRequest) Stream~RecordBatch~
  }
  class AdapterRegistry {
    +register(name, Adapter)
    +route(schema, table) Adapter
    +schemas() Vec~SchemaDef~
  }
  class ManifestAdapter {
    -manifest: Manifest
    +catalog()
    +scan()
  }
  class PolymarketAdapter {
    -inner: ManifestAdapter
    +catalog()
    +scan()
  }
  class HttpClient {
    +fetch_paginated(req) Stream~Bytes~
  }
  class Manifest {
    +source: SourceCfg
    +tables: Vec~TableCfg~
  }
  Adapter <|.. ManifestAdapter
  Adapter <|.. PolymarketAdapter
  PolymarketAdapter o-- ManifestAdapter
  ManifestAdapter --> Manifest
  ManifestAdapter --> HttpClient
  AdapterRegistry o-- Adapter
```

## 10. Integration architecture (grounded in current `spur-notebook` code)

This is the part where the new gateway meets the **existing** datasource plumbing. Each box below cites the real symbol and `file:line` verified via the `code_*` graph (see *Grounding Appendix*).

### The key mismatch to resolve
Every existing datasource is a **file path**: `attach_datasource` does `normalize_datasource_path` -> `infer_datasource_kind(&path)` -> `introspect_datasource(&path, kind)` (`crates/spur-notebook/src/mcp/mod.rs:1085`). An **airport** datasource has **no file path** — its locator is `{ source_name, gateway_addr }`. So we add a parallel attach route rather than overloading path inference.

```mermaid
flowchart TB
  subgraph existing["EXISTING (today) — path-based datasources"]
    a1["attach_datasource(name, path, group)\nmcp/mod.rs:1085"]
    a2["normalize_datasource_path()"]
    a3["infer_datasource_kind(&path) (extension-based)"]
    a4["introspect_datasource(&path, kind)\ndatasource/mod.rs:24"]
    a5["DuckDB probe: read_csv/parquet/json or ATTACH (duckdb/sqlite)"]
    a1 --> a2 --> a3 --> a4 --> a5
  end

  subgraph new["NEW — airport datasource"]
    b1["attach_airport_datasource(name, source, gateway_addr) (new daemon method)"]
    b2["ensure gateway running (GatewayHandle::spawn, loopback+token)"]
    b3["introspect_datasource locator=Airport (extend datasource/mod.rs)"]
    b4["DuckDB: ATTACH (TYPE airport, location grpc://127.0.0.1:PORT) + airport_list_flights / DESCRIBE"]
    b1 --> b2 --> b3 --> b4
  end

  subgraph shared["SHARED catalog surface (unchanged contract)"]
    c1["DatasourceEntry { name, locator, kind=Airport, group, columns, row_count, tables }\ncommands.rs:163 + acp events.rs:124"]
    c2["refresh_datasource_setup_cell()"]
    c3["persist_catalog_to_current_notebook()"]
    c4["reconcile_open_datasource_catalog() mcp/mod.rs:1611 (re-probe on open)"]
    c5["list_datasources MCP tool tools/mod.rs:19 -> UI panel"]
  end

  a5 --> c1
  b4 --> c1
  c1 --> c2 --> c3
  c4 -. "re-introspects empty entries (now also airport)" .-> b3
  c1 --> c5
  b2 -. "on notebook open, if any Airport entry" .-> c4
```

### 10b. Attach & reconcile sequence (airport datasource)

Grounded in `attach_datasource` (`mcp/mod.rs:1085`) and `reconcile_open_datasource_catalog` (`mcp/mod.rs:1611`), which today re-probes any entry whose `columns`/`tables` are empty — the natural hook to (re)introspect the airport catalog on notebook open.

```mermaid
sequenceDiagram
  participant UI as Notebook UI
  participant D as NotebookDaemonControl
  participant G as GatewayHandle
  participant Duck as DuckDB+airport
  participant J as jute datasource catalog

  Note over UI,J: Attach (new airport source)
  UI->>D: attach_airport_datasource(name, source, addr?)
  D->>G: ensure_running() -> grpc://127.0.0.1:PORT + token
  G-->>D: GatewayHandle{ addr, token }
  D->>Duck: ATTACH source (TYPE airport, location addr)
  D->>Duck: airport_list_flights + DESCRIBE per table
  Duck-->>D: columns / tables
  D->>J: attach_datasource(DatasourceEntry{kind:Airport})
  D->>D: refresh_datasource_setup_cell + persist_catalog

  Note over UI,J: Reopen notebook later
  UI->>D: notebook open
  D->>D: reconcile_open_datasource_catalog() (mod.rs:1611)
  alt entry.columns empty AND kind==Airport
    D->>G: ensure_running()
    D->>Duck: re-ATTACH + re-introspect
    Duck-->>D: fresh schema
    D->>J: attach_datasources(reconciled)
  end
```

## 11. Type & contract changes

| Change | File (grounded) | Note |
|---|---|---|
| Add `Airport` variant to `DatasourceKind` | `jute-notebook/src-tauri/src/commands.rs:121` | canonical enum; `#[ts]` regenerates the TS binding |
| Mirror `Airport` in the ACP event enum | `crates/spur-acp/src/domain/events.rs:96` | second copy kept in sync (wire contract) |
| Regenerate TS binding | `jute-notebook/src/bindings/DatasourceKind.ts` | via `ts-rs` (already in deps) |
| Extend datasource **locator** beyond `path` | `DatasourceEntry` `commands.rs:163` + `events.rs:124` | `path: String` no longer fits airport; introduce a `locator` (enum: `Path(String)` or `Airport{source, addr}`) or keep `path` + add optional `gateway_addr`. **Decision needed — see Open Questions.** |
| Airport branch in `introspect_datasource` | `crates/spur-notebook/src/datasource/mod.rs:24` | `ATTACH (TYPE airport)` then enumerate via the existing DuckDB probe path used for `DuckDb`/`Sqlite` |
| New daemon attach route | `crates/spur-notebook/src/mcp/mod.rs` near `attach_datasource:1085` | bypasses path inference; ensures gateway is running |

> **Wire-compat caution:** `DatasourceKind` and `DatasourceEntry` are duplicated across `jute::commands` and `spur-acp::domain::events` and exported to TS. Any field change must update **all three** in one change to keep the bridge contract intact — there's an existing `datasource_wire_contract` test guarding this.

## 12. Auth & secrets

Two trust boundaries: (1) DuckDB<->gateway is loopback-only with a startup Bearer token (defense-in-depth on localhost); (2) gateway<->upstream uses per-source credentials resolved **server-side** and never returned to the SQL client.

```mermaid
flowchart LR
  duck[("DuckDB + airport")] -- "Bearer startup-token (loopback only)" --> hs["Handshake / interceptor"]
  hs --> svc["FlightService"]
  svc --> sec["secrets.rs resolve per-source creds"]
  sec -- "env / secret store" --> store[("credential source")]
  sec -- "ResolvedAuth (never leaves process)" --> adp["Adapter::scan"]
  adp -- "signed/authd request" --> api[["upstream API"]]
  api -. "API key NEVER returned to client" .-x duck
```

## 13. Error handling

- **Upstream / adapter errors** -> mapped to a typed `tonic::Status`; surfaced as a clean DuckDB error string in the cell (no opaque gRPC noise).
- **Schema drift** (API returns unexpected JSON) -> declared-schema typed coercion with a precise `field X expected T, got ...` error. **Never** silent null-fill.
- **Rate limits / HTTP 429** -> framework-level retry-with-backoff in `http.rs`; 429 responses are **not** cached.
- **Gateway unavailable at attach** -> mirrors the existing `attach_datasource` behavior: log a warning and attach with **pending** (empty) metadata, then fill in on `reconcile_open_datasource_catalog`.
- **Partial stream failure mid-`DoGet`** -> the batch stream yields an error item; DuckDB surfaces it; no partial-result silent truncation.

## 14. Testing strategy

| Layer | What | How |
|---|---|---|
| **Wire port (risk #1)** | `ticket.rs` / `filter.rs` round-trips | Golden fixtures captured from `airport-go`; encode->decode equality |
| **Adapter contract** | pagination, predicate->param mapping, schema coercion, TVF args | `wiremock` fake REST server — deterministic, offline |
| **JSON->Arrow** | typed coercion, drift errors | unit tests on `arrow_json.rs` |
| **End-to-end** | real `duckdb` crate + airport ext, `ATTACH` + `SELECT ... WHERE` pushdown, assert RecordBatches | in-process gateway; mirrors existing `datasource_wire_contract` + `attach_duckdb_introspects_all_tables` test style (`datasource/mod.rs:172`) |
| **Live Polymarket** | smoke test against real API | `#[ignore]` / manual — never in CI |

The end-to-end test reuses the established pattern of the existing `introspect_datasource` tests (`attach_duckdb_introspects_all_tables` @ `datasource/mod.rs:172`, `attach_analyst_index_introspects_tables_and_views` @ `:228`).

## 15. Proof — Polymarket as adapter #1

**Manifest table + pushdown:**
```sql
SELECT question, volume
FROM polymarket.markets
WHERE active = true        -- pushed down to ?active=true
ORDER BY volume DESC       -- residual, applied by DuckDB
LIMIT 10;
```

**TVF via trait escape hatch:**
```sql
SELECT price, size
FROM polymarket.orderbook('0xabc...token', depth => 50);  -- args -> path/query
```

Polymarket validates **both** halves of C3 in one adapter: the manifest handles `markets` / `events`; a ~20-line trait impl handles the parameterized `orderbook` TVF that a pure manifest cannot express.

## 16. Phasing / milestones

```mermaid
flowchart LR
  M0["M0 Skeleton: crate + tonic Flight server, Handshake + ListFlights (hello-world table)"]
  M1["M1 Wire port: ticket.rs + filter.rs from airport-go + golden tests"]
  M2["M2 Manifest path: ManifestAdapter + http.rs + arrow_json, DoGet streaming + pushdown"]
  M3["M3 Notebook integration: Airport DatasourceKind + locator, attach route + reconcile + introspect"]
  M4["M4 Polymarket: manifest tables + orderbook TVF, end-to-end test"]
  M5["M5 Hardening: rate-limit + cache + retry, error mapping + docs"]
  M0 --> M1 --> M2 --> M3 --> M4 --> M5
```

M0-M2 are gateway-internal (no notebook changes). M3 is the only milestone that edits shared `spur-notebook` contracts and must land the three enum mirrors + wire-contract test together.

## 17. Decisions (all recommendations approved 2026-05-31 — move forward, refine later)

**Status:** All five recommendations are **accepted** for v1, exactly as the *Recommendation* lines stated. Each has a clear later-refinement path; revisit during/after M3 if needed.

1. **Locator shape.** ✅ **Additive optional fields** for v1 — keep `DatasourceEntry.path` and add optional `gateway_addr` + `source`. Minimizes wire-contract churn across the three mirrors + TS binding. Enum (`Path` | `Airport`) refactor deferred.
2. **Gateway lifecycle.** ✅ **Lazy** — `GatewayHandle::ensure_running` on first airport attach, reused across sources.
3. **Manifest format.** ✅ **TOML** (matches Rust/Cargo ergonomics). Revisit if adapters get authored by non-Rust users.
4. **Manifest/adapter discovery dir.** ✅ A **single configured directory, global** for v1 (not per-notebook).
5. **In-process only.** ✅ **Yes** — no standalone gateway binary in v1; the notebook backend owns the gateway lifecycle in-process.

> These are intentionally the lowest-churn choices so M0–M4 proceed without contract thrash. "Refine later" is explicitly in scope after the end-to-end Polymarket path works (post-M4).

## 18. Grounding appendix (verified via `code_*` graph)

Graph content hash `f76ef47d...`, worktree head `8a6522e0...`, `response_file_oids_match: true` at authoring time.

| Symbol | Kind | file:line | stable id |
|---|---|---|---|
| `DatasourceKind` (canonical) | enum | `crates/spur-notebook/jute-notebook/src-tauri/src/commands.rs:121` | `fc0a4090d5e2c116` |
| `DatasourceKind` (ACP mirror) | enum | `crates/spur-acp/src/domain/events.rs:96` | `4d960237dd35e917` |
| `DatasourceEntry` (canonical) | struct | `crates/spur-notebook/jute-notebook/src-tauri/src/commands.rs:163` | `1e8e7e06ee714fe5` |
| `DatasourceEntry` (ACP mirror) | struct | `crates/spur-acp/src/domain/events.rs:124` | `54e439f9335fe5ea` |
| `DatasourceEntry` (TS binding) | type | `crates/spur-notebook/jute-notebook/src/bindings/DatasourceEntry.ts:9` | `32103eb38290b625` |
| `introspect_datasource` | fn | `crates/spur-notebook/src/datasource/mod.rs:24` | `2fc20d846ceee96f` |
| `attach_datasource` | method | `crates/spur-notebook/src/mcp/mod.rs:1085` | `7ba96d0162084f4e` |
| `reconcile_open_datasource_catalog` | method | `crates/spur-notebook/src/mcp/mod.rs:1611` | `ded9a6e26533f1db` |
| `list_datasources` (MCP tool mod) | module | `crates/spur-notebook/src/mcp/tools/mod.rs:19` | `239ea0af390fa441` |
| `attach_duckdb_introspects_all_tables` (test pattern) | fn | `crates/spur-notebook/src/datasource/mod.rs:172` | `366150a1fdbe0043` |

**Callers of `introspect_datasource`** (the seam the airport branch joins): `attach_datasource` (`mod.rs:1085`), `reconcile_open_datasource_catalog` (`mod.rs:1611`), plus the two introspection tests. No other call sites — blast radius of the airport branch is contained to these two daemon methods + the probe function.

### External references
- airport extension: <https://airport.query.farm/> · features: <https://airport.query.farm/features.html>
- `airport-go` (wire-contract reference to port): <https://pkg.go.dev/github.com/hugr-lab/airport-go>
- `python-flight-server`: <https://github.com/Query-farm/python-flight-server>
- `arrow-flight` crate: <https://docs.rs/arrow-flight/latest/arrow_flight/>